# ChartNarrator Stage 2 text-only fine-tuning (WA >= 4.5)

Public cleanup of the original `train_0320_textonly_4_5` notebook. The training and inference logic is preserved, while local absolute paths, Chinese comments/output text, private Colab paths, and long execution outputs have been removed.

This notebook corresponds to the text-only ablation. It uses the same Stage 2 train/validation/test split as the main WA >= 4.5 VLM condition, but removes the image input. The user prompt contains the serialized time-series representation and the structured `[CHART CONTEXT]` anchor text.

Original run record from the cleaned notebook outputs:

- Model: `Qwen/Qwen2.5-7B-Instruct`
- Dataset split: train = 986, validation = 124, test = 124
- Fine-tuning: QLoRA, 4-bit NF4, LoRA rank = 64, alpha = 32, epochs = 8
- Training config: `eval_strategy = "no"` to avoid text-only evaluation-time OOM in the saved run environment
- The saved training cell output contains a runtime traceback, so final train/eval loss is not preserved in this notebook output
- Test inference: 124 / 124 successful
- Format compliance: P1 = 100%, P2 = 100%, P3 = 100%, full format compliance = 100.0%


## 1. Configure repository paths

This cell locates the repository root, defines the text-only dataset, checkpoint, and prediction-output paths, and prints split counts when the data is already present.


In [ ]:
import json
import os
from pathlib import Path

# Set CHARTNARRATOR_ROOT when running outside the repository root.
# Example: os.environ["CHARTNARRATOR_ROOT"] = "/content/ChartNarrator_public"
def resolve_project_root():
    env_root = os.environ.get("CHARTNARRATOR_ROOT")
    if env_root:
        return Path(env_root).resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "data").exists() or (candidate / "scripts").exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "finetune_textonly_4_5"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "qwen25_textonly_chartnarrator_stage2_4_5"
PREDICTION_OUTPUT = PROJECT_ROOT / "data" / "evaluation" / "predictions" / "predictions_test_textonly_stage2_4_5_round3.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATA_DIR}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Predictions  : {PREDICTION_OUTPUT}")

for split in ["train", "val", "test"]:
    split_path = DATA_DIR / f"{split}.json"
    if split_path.exists():
        with open(split_path, encoding="utf-8") as f:
            print(f"{split}.json: {len(json.load(f))} entries")
    else:
        print(f"{split}.json: missing")


## 2. Inspect one text-only training example

This optional sanity check previews the serialized text input and target response. It verifies that the prompt contains `[TIME SERIES DATA]` and `[CHART CONTEXT]`, while the image field is absent.


In [ ]:
import json

with open(DATA_DIR / "train.json", encoding="utf-8") as f:
    train = json.load(f)

e = train[0]
user_prompt = e["conversations"][0]["value"]

print("=== user prompt preview (first 500 chars) ===")
print(user_prompt[:500])
print()
print("Contains [TIME SERIES DATA]:", "[TIME SERIES DATA]" in user_prompt)
print("Contains [CHART CONTEXT]:", "[CHART CONTEXT]" in user_prompt)
print()
print("=== assistant target preview (first 200 chars) ===")
print(e["conversations"][1]["value"][:200])
print()
print("=== image field ===")
print(e.get("image", "NO IMAGE FIELD"))


## 3. Verify or install the training environment

This cell checks the expected PyTorch, Transformers, Accelerate, and LLaMA-Factory versions. Installation is only triggered when the stack is missing or incompatible.


In [ ]:
import importlib
import os
import sys


def check_versions():
    """Return True when the required training stack is already installed."""
    try:
        import torch
        import transformers
        import accelerate
        import llamafactory
        ok = (
            torch.__version__.startswith("2.5.1")
            and transformers.__version__ == "4.46.1"
            and accelerate.__version__ == "1.0.1"
        )
        if ok:
            print("Training environment is ready; installation skipped.")
            print(
                f"PyTorch {torch.__version__} | "
                f"Transformers {transformers.__version__} | "
                f"Accelerate {accelerate.__version__}"
            )
            return True
    except ImportError:
        pass
    return False


if check_versions():
    pass
else:
    print("Removing incompatible packages...")
    os.system("pip uninstall -y torch torchvision torchaudio accelerate transformer-engine flash-attn")

    print("Installing PyTorch 2.5.1...")
    os.system(
        "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 "
        "--index-url https://download.pytorch.org/whl/cu121"
    )

    print("Installing training dependencies...")
    os.system("pip install transformers==4.46.1 accelerate==1.0.1 peft==0.12.0 bitsandbytes==0.44.1")
    os.system("pip install datasets==2.21.0 trl==0.9.6 scipy einops sentencepiece protobuf tiktoken")
    os.system("pip install deepspeed==0.15.4")
    os.system("pip install flash-attn==2.7.4.post1 --no-build-isolation")

    print("Installing LLaMA-Factory v0.9.1...")
    os.system("git clone --depth 1 -b v0.9.1 https://github.com/hiyouga/LLaMA-Factory.git")
    os.chdir("LLaMA-Factory")
    os.system("pip install -e .[torch,metrics]")
    os.chdir("..")

    print("Installation finished. Restart the runtime, then continue from the next cell.")


## 4. Write `dataset_info.json`

This cell registers the text-only train, validation, and test split names in the LLaMA-Factory ShareGPT format. Unlike VLM notebooks, it does not declare an image column.


In [ ]:
import json

INFO_PATH = DATA_DIR / "dataset_info.json"

dataset_info = {}
for split in ["train", "val", "test"]:
    dataset_info[f"textonly_4_5_{split}"] = {
        "file_name": f"{split}.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "conversations",
        },
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "user",
            "assistant_tag": "assistant",
        },
    }

with open(INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"dataset_info.json written to: {INFO_PATH}")
for key in dataset_info:
    print(f"dataset key: {key}")


## 5. Write the text-only QLoRA training configuration

This cell creates the LLaMA-Factory YAML config. It preserves the text-only model, 4-bit quantization, LoRA parameters, batch settings, and the original OOM workaround that disables training-time evaluation.


In [ ]:
import json
import os
import yaml

split_counts = {}
for split in ["train", "val", "test"]:
    split_path = DATA_DIR / f"{split}.json"
    if split_path.exists():
        split_counts[split] = len(json.loads(split_path.read_text(encoding="utf-8")))
    else:
        split_counts[split] = "missing"

train_args = {
    "model_name_or_path": "Qwen/Qwen2.5-7B-Instruct",
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "quantization_bit": 4,
    "template": "qwen",

    "optim": "paged_adamw_32bit",
    "gradient_checkpointing": True,

    "dataset_dir": str(DATA_DIR),
    "dataset": "textonly_4_5_train",

    "cutoff_len": 4096,
    "learning_rate": 1e-5,
    "num_train_epochs": 8.0,

    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,

    "eval_strategy": "no",
    "save_steps": 200,
    "logging_steps": 5,
    "save_total_limit": 3,

    "lora_rank": 64,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target": "all",

    "output_dir": str(CHECKPOINT_DIR),
    "overwrite_output_dir": True,
    "plot_loss": True,
    "bf16": True,
    "fp16": False,
    "ddp_timeout": 180000000,
}

config_path = PROJECT_ROOT / "train_config_textonly_4_5.yaml"
with open(config_path, "w") as f:
    yaml.dump(train_args, f)

print(f"Training config written to: {config_path}")
print(
    "Dataset: "
    f"train={split_counts['train']} | val={split_counts['val']} | test={split_counts['test']}"
)
print("LoRA: rank=64 | alpha=32 | epochs=8")
print("Input: serialized time-series data + anchor text, without chart image")
print("OOM-related setting: eval_strategy='no'; run prediction/evaluation after training")
print(f"Output checkpoint dir: {CHECKPOINT_DIR}")


## 6. Launch fine-tuning

This long-running cell starts text-only QLoRA fine-tuning from the generated YAML config. Run it only in a GPU environment with the required model and dataset files available.


In [ ]:
import os
import torch

torch.cuda.empty_cache()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Stage 2 text-only training start")
print("Expected paper split: train=986 | val=124 | test=124")
print("Model: Qwen2.5-7B-Instruct")
print("LoRA rank=64 | alpha=32 | epochs=8")
print("Training-time evaluation is disabled to avoid OOM in the saved run setup.")
!llamafactory-cli train {str(config_path)}


## 7. Run text-only test-set inference

This cell loads the 4-bit Qwen2.5 base model, attaches the LoRA adapter, applies the Qwen chat template to the serialized text prompt, and writes test predictions.


In [ ]:
import json
import os

import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

TEST_FILE = DATA_DIR / "test.json"
PREDICTION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("ChartNarrator Stage 2 text-only inference")
print("=" * 70)
print(f"Checkpoint : {CHECKPOINT_DIR}")
print(f"Test set   : {TEST_FILE}")
print(f"Output     : {PREDICTION_OUTPUT}")
print("=" * 70)

print("Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="cuda",
)
model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_DIR))
model.eval()

model.generation_config = GenerationConfig(
    bos_token_id=151643,
    eos_token_id=151645,
    pad_token_id=151643,
)

tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT_DIR))
print("Model loaded.")

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test samples: {len(test_data)}")

predictions = []

for idx, entry in enumerate(tqdm(test_data, desc="Inference")):
    try:
        user_query = entry["conversations"][0]["value"].strip()

        messages = [
            {"role": "user", "content": user_query}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False,
                repetition_penalty=1.1,
            )

        generated_ids_trimmed = generated_ids[0][inputs.input_ids.shape[1]:]
        output_text = tokenizer.decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        has_p1 = "── Paragraph 1" in output_text
        has_p2 = "── Paragraph 2" in output_text
        has_p3 = "── Paragraph 3" in output_text

        predictions.append(
            {
                "id": entry["id"],
                "morphology_family": entry.get("morphology_family", ""),
                "route_label": entry.get("route_label", ""),
                "conversations": [
                    {"from": "user", "value": user_query},
                    {"from": "assistant", "value": output_text},
                ],
                "reference": entry["conversations"][1]["value"],
                "format_check": {"p1": has_p1, "p2": has_p2, "p3": has_p3},
            }
        )

        if idx == 0:
            print("\n" + "=" * 70)
            print(f"First sample preview ({entry['id']})")
            print("=" * 70)
            print(output_text[:600])
            print("=" * 70 + "\n")

    except Exception as e:
        print(f"Sample {idx} ({entry.get('id', '?')}) failed: {e}")
        predictions.append({"id": entry.get("id", ""), "error": str(e)})

with open(PREDICTION_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

successful = [p for p in predictions if "error" not in p]
print("\n" + "=" * 70)
print("Inference statistics")
print("=" * 70)
print(f"Success: {len(successful)} / {len(predictions)}")
if successful:
    p1 = sum(1 for p in successful if p["format_check"]["p1"]) / len(successful) * 100
    p2 = sum(1 for p in successful if p["format_check"]["p2"]) / len(successful) * 100
    p3 = sum(1 for p in successful if p["format_check"]["p3"]) / len(successful) * 100
    all_fmt = sum(1 for p in successful if all(p["format_check"].values())) / len(successful) * 100
    print(f"Format: P1={p1:.0f}% | P2={p2:.0f}% | P3={p3:.0f}% | full={all_fmt:.1f}%")
print(f"Results saved to: {PREDICTION_OUTPUT}")


## 8. Optional checkpoint sanity check

This optional verification cell lists checkpoint files and checks whether `adapter_config.json` exists. It is useful before running inference but is not part of the experimental method itself.


In [ ]:
import os

print("Files in checkpoint dir:")
if CHECKPOINT_DIR.exists():
    for file_name in os.listdir(CHECKPOINT_DIR):
        print(f" {file_name}")
    print("\nadapter_config.json exists:", (CHECKPOINT_DIR / "adapter_config.json").exists())
else:
    print(f"Checkpoint directory not found: {CHECKPOINT_DIR}")
